In [1]:
import pandas as pd
import random
import os
import accelerate
import torch
from unsloth import FastLanguageModel, get_chat_template
import re
import gc

c:\PokeAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1119 21:43:40.938000 13600 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\PokeAI\.venv\Lib\site-packages\peft\import_utils.py:52: FutureWarning: Support for Transformers < 5.6 is deprecated in PEFT and will be removed in the next release, v1.0. You have Transformers 5.5.0. Please upgrade to Transformers >= 5.6.
  warnings.warn(


🦥 Unsloth Zoo will now patch everything to make training faster!


In [7]:
models = [
    #'DeepSeek-V4-Pro-Qwen3.5-4B',
    #'gemma-3-4b-it',
    #'Llama-3.2-3B-Instruct',
    'Qwen3.5-4B'
]
sistema = '''
Você é o assistente do PokeAI, especializado em Metagame de Pokémon competitivo. 
Responda em português, com clareza e precisão. 
'''
pergunta = "Qual é o Pokémon com maior taxa de uso absoluto no metagame VGC 2026 Regulation M-B em alto elo (1760+ Elo, Bo3)?"

prompt = f"{sistema}\n\n{pergunta}"
messages = [{"role": "user", "content": prompt}]

os.environ["UNSLOTH_DISABLE_FAST_GENERATION"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

for model_name in models:
    try:
        print(f"\n\nCarregando modelo: {model_name}\n\n")
        path = f'C:/PokeAI/Models/{model_name}'
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=path,
            max_seq_length=4096,
            #dtype=None,
            load_in_4bit=True,
            device_map="cuda:0",
        )

        model.generation_config.max_length = None
        FastLanguageModel.for_inference(model)
        #if model_name in ['DeepSeek-V4-Pro-Qwen3.5-4B', 'Qwen3.5-4B']:
            #tokenizer = get_chat_template(tokenizer, chat_template='chatml')

        if tokenizer.pad_token_id is None:
            tokenizer.pad_token_id = tokenizer.eos_token_id


        print(f"Modelo {model_name} carregado com sucesso. Iniciando inferência...")
        if model_name == 'Qwen3.5-4B':
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        else:
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        #inputs = tokenizer([prompt], return_tensors="pt", padding=True).to("cuda")
        inputs = tokenizer(text=[prompt], return_tensors="pt", padding=True).to("cuda")
        prompt_len = inputs.input_ids.shape[1]

        outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, do_sample=False)#, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)

        generated_tokens = outputs[0][prompt_len:]
        hypothesis = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        print(f"\n\nResposta COMPLETA: {hypothesis}\n{'-'*80}")
        if "</think>" in hypothesis:
            hypothesis = hypothesis.split("</think>", 1)[1].strip()
        print(f"\n\nResposta FINAL: {hypothesis}\n{'-'*80}")

        print(f"Finalizando modelo: {model_name}\n\n")

        del model
        del tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    except Exception as e:
        print(f"Erro ao processar o modelo {model_name}: {e}")
        try:
            del model
            del tokenizer
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            continue
        except:
            continue
# Qwen papagaio



Carregando modelo: Qwen3.5-4B


==((====))==  Unsloth 2026.9.4: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce GTX 1660 SUPER. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.9.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights: 100%|██████████| 723/723 [00:42<00:00, 16.83it/s]


Modelo Qwen3.5-4B carregado com sucesso. Iniciando inferência...


Resposta COMPLETA: Atualmente, **não existe** um Pokémon com maior taxa de uso absoluto no metagame VGC 2026 Regulation M-B em alto elo (1760+ Elo, Bo3), pois **o formato Regulation M-B não foi implementado** e, consequentemente, não há dados de torneios ou rankings oficiais para ele.

Aqui estão os detalhes importantes para esclarecer a situação:

1.  **Status do Regulation M-B**: O formato M-B foi anunciado como uma expansão futura para o VGC, mas até o momento (fim de 2024/início de 2025), ele **não está ativo** em nenhum torneio oficial da Nintendo ou da comunidade global. Sem torneios ocorrendo, não há dados de "High Elo" ou "Bo3" para este formato.
2.  **Metagame Atual (Regulation M-A)**: O formato ativo e competitivo é o **Regulation M-A**. Se você estiver procurando por dados reais de alto elo no VGC atual, o Pokémon com maior taxa de uso absoluto é historicamente o **Garchomp** (ou **Garchomp-EX** em contextos 

In [ ]:
models = [
    'DeepSeek-V4-Pro-Qwen3.5-4B',
    #'gemma-3-4b-it',
    #'Llama-3.2-3B-Instruct',
    #'Qwen3.5-4B'
]
sistema = '''
Você é o assistente do PokeAI, especializado em Metagame de Pokémon competitivo. 
Responda em português, com clareza e precisão. 
'''
pergunta = "Qual é o Pokémon com maior taxa de uso absoluto no metagame VGC 2026 Regulation M-B em alto elo (1760+ Elo, Bo3)?"

prompt = f"{sistema}\n\n{pergunta}"
messages = [{"role": "user", "content": prompt}]

os.environ["UNSLOTH_DISABLE_FAST_GENERATION"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

for model_name in models:
    try:
        print(f"\n\nCarregando modelo: {model_name}\n\n")
        path = f'C:/PokeAI/Models/{model_name}'
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=path,
            max_seq_length=4096,
            #dtype=None,
            load_in_4bit=True,
            device_map="cuda:0",
        )

        model.generation_config.max_length = None
        FastLanguageModel.for_inference(model)
        #if model_name in ['DeepSeek-V4-Pro-Qwen3.5-4B', 'Qwen3.5-4B']:
            #tokenizer = get_chat_template(tokenizer, chat_template='chatml')

        if tokenizer.pad_token_id is None:
            tokenizer.pad_token_id = tokenizer.eos_token_id


        print(f"Modelo {model_name} carregado com sucesso. Iniciando inferência...")
        if model_name == 'Qwen3.5-4B':
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        else:
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        #inputs = tokenizer([prompt], return_tensors="pt", padding=True).to("cuda")
        inputs = tokenizer(text=[prompt], return_tensors="pt", padding=True).to("cuda")
        prompt_len = inputs.input_ids.shape[1]

        outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, do_sample=False)#, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)

        generated_tokens = outputs[0][prompt_len:]
        hypothesis = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        print(f"\n\nResposta COMPLETA: {hypothesis}\n{'-'*80}")
        if "</think>" in hypothesis:
            hypothesis = hypothesis.split("</think>", 1)[1].strip()
        print(f"\n\nResposta FINAL: {hypothesis}\n{'-'*80}")

        print(f"Finalizando modelo: {model_name}\n\n")

        del model
        del tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    except Exception as e:
        print(f"Erro ao processar o modelo {model_name}: {e}")
        try:
            del model
            del tokenizer
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            continue
        except:
            continue
# Qwen papagaio



Carregando modelo: DeepSeek-V4-Pro-Qwen3.5-4B


==((====))==  Unsloth 2026.9.4: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce GTX 1660 SUPER. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.9.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights: 100%|██████████| 723/723 [00:46<00:00, 15.48it/s]
The tokenizer you are loading from 'C:/PokeAI/Models/DeepSeek-V4-Pro-Qwen3.5-4B' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Modelo DeepSeek-V4-Pro-Qwen3.5-4B carregado com sucesso. Iniciando inferência...


In [ ]:
models = [
    #'DeepSeek-V4-Pro-Qwen3.5-4B',
    #'gemma-3-4b-it',
    'Llama-3.2-3B-Instruct',
    #'Qwen3.5-4B'
]
sistema = '''
Você é o assistente do PokeAI, especializado em Metagame de Pokémon competitivo. 
Responda em português, com clareza e precisão. 
'''
pergunta = "Qual é o Pokémon com maior taxa de uso absoluto no metagame VGC 2026 Regulation M-B em alto elo (1760+ Elo, Bo3)?"

prompt = f"{sistema}\n\n{pergunta}"
messages = [{"role": "user", "content": prompt}]

os.environ["UNSLOTH_DISABLE_FAST_GENERATION"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

for model_name in models:
    try:
        print(f"\n\nCarregando modelo: {model_name}\n\n")
        path = f'C:/PokeAI/Models/{model_name}'
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=path,
            max_seq_length=4096,
            #dtype=None,
            load_in_4bit=True,
            device_map="cuda:0",
        )

        model.generation_config.max_length = None
        FastLanguageModel.for_inference(model)
        #if model_name in ['DeepSeek-V4-Pro-Qwen3.5-4B', 'Qwen3.5-4B']:
            #tokenizer = get_chat_template(tokenizer, chat_template='chatml')

        if tokenizer.pad_token_id is None:
            tokenizer.pad_token_id = tokenizer.eos_token_id


        print(f"Modelo {model_name} carregado com sucesso. Iniciando inferência...")
        if model_name == 'Qwen3.5-4B':
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        else:
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        #inputs = tokenizer([prompt], return_tensors="pt", padding=True).to("cuda")
        inputs = tokenizer(text=[prompt], return_tensors="pt", padding=True).to("cuda")
        prompt_len = inputs.input_ids.shape[1]

        outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, do_sample=False)#, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)

        generated_tokens = outputs[0][prompt_len:]
        hypothesis = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        print(f"\n\nResposta COMPLETA: {hypothesis}\n{'-'*80}")
        if "</think>" in hypothesis:
            hypothesis = hypothesis.split("</think>", 1)[1].strip()
        print(f"\n\nResposta FINAL: {hypothesis}\n{'-'*80}")

        print(f"Finalizando modelo: {model_name}\n\n")

        del model
        del tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    except Exception as e:
        print(f"Erro ao processar o modelo {model_name}: {e}")
        try:
            del model
            del tokenizer
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            continue
        except:
            continue
# Qwen papagaio

In [ ]:
models = [
    #'DeepSeek-V4-Pro-Qwen3.5-4B',
    'gemma-3-4b-it',
    #'Llama-3.2-3B-Instruct',
    #'Qwen3.5-4B'
]
sistema = '''
Você é o assistente do PokeAI, especializado em Metagame de Pokémon competitivo. 
Responda em português, com clareza e precisão. 
'''
pergunta = "Qual é o Pokémon com maior taxa de uso absoluto no metagame VGC 2026 Regulation M-B em alto elo (1760+ Elo, Bo3)?"

prompt = f"{sistema}\n\n{pergunta}"
messages = [{"role": "user", "content": prompt}]

os.environ["UNSLOTH_DISABLE_FAST_GENERATION"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

for model_name in models:
    try:
        print(f"\n\nCarregando modelo: {model_name}\n\n")
        path = f'C:/PokeAI/Models/{model_name}'
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=path,
            max_seq_length=4096,
            #dtype=None,
            load_in_4bit=True,
            device_map="cuda:0",
        )

        model.generation_config.max_length = None
        FastLanguageModel.for_inference(model)
        #if model_name in ['DeepSeek-V4-Pro-Qwen3.5-4B', 'Qwen3.5-4B']:
            #tokenizer = get_chat_template(tokenizer, chat_template='chatml')

        if tokenizer.pad_token_id is None:
            tokenizer.pad_token_id = tokenizer.eos_token_id


        print(f"Modelo {model_name} carregado com sucesso. Iniciando inferência...")
        if model_name == 'Qwen3.5-4B':
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        else:
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        #inputs = tokenizer([prompt], return_tensors="pt", padding=True).to("cuda")
        inputs = tokenizer(text=[prompt], return_tensors="pt", padding=True).to("cuda")
        prompt_len = inputs.input_ids.shape[1]

        outputs = model.generate(**inputs, max_new_tokens=2048, use_cache=True, do_sample=False)#, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)

        generated_tokens = outputs[0][prompt_len:]
        hypothesis = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        print(f"\n\nResposta COMPLETA: {hypothesis}\n{'-'*80}")
        if "</think>" in hypothesis:
            hypothesis = hypothesis.split("</think>", 1)[1].strip()
        print(f"\n\nResposta FINAL: {hypothesis}\n{'-'*80}")

        print(f"Finalizando modelo: {model_name}\n\n")

        del model
        del tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    except Exception as e:
        print(f"Erro ao processar o modelo {model_name}: {e}")
        try:
            del model
            del tokenizer
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            continue
        except:
            continue
# Qwen papagaio